In [18]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("../dataset/resumes.csv")
print(df.head())
x = df["resume"]
y = df["job_role"]

vectorizer = TfidfVectorizer()
x_tfidf = vectorizer.fit_transform(x)
print(x_tfidf.shape)

x_train, x_test, y_train, y_test = train_test_split(
    x_tfidf,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)
    
model = LogisticRegression()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

print("Actual:")
print(y_test.values)

print("\nPredicted:")
print(y_pred)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print(classification_report(y_test, y_pred, zero_division=0))

# Test with a completely new resume
new_resume = [
    "Python Pandas NumPy SQL Excel data analysis"
]
new_resume_tfidf = vectorizer.transform(new_resume)
prediction = model.predict(new_resume_tfidf)
print("Predicted Job Role:", prediction[0])

# Test Python Developer
new_resume = [
    "Python Django Flask SQL Git backend API development"
]
new_resume_tfidf = vectorizer.transform(new_resume)
prediction = model.predict(new_resume_tfidf)
print("Predicted Job Role:", prediction[0])

# Test Web Developer
new_resume = [
    "HTML CSS JavaScript React frontend website development"
]
new_resume_tfidf = vectorizer.transform(new_resume)
prediction = model.predict(new_resume_tfidf)
print("Predicted Job Role:", prediction[0])




                                              resume          job_role
0   python pandas numpy sql data analysis statistics      Data Analyst
1  python pandas excel sql visualization data ana...      Data Analyst
2      python numpy pandas matplotlib sql statistics      Data Analyst
3                   python sql pandas power bi excel      Data Analyst
4            python django flask sql git api backend  Python Developer
(20, 46)
Actual:
<StringArray>
[     'ML Engineer',     'Data Analyst',    'Web Developer',
   'Java Developer', 'Python Developer']
Length: 5, dtype: str

Predicted:
['ML Engineer' 'Data Analyst' 'Web Developer' 'Java Developer'
 'Python Developer']
Accuracy: 1.0
                  precision    recall  f1-score   support

    Data Analyst       1.00      1.00      1.00         1
  Java Developer       1.00      1.00      1.00         1
     ML Engineer       1.00      1.00      1.00         1
Python Developer       1.00      1.00      1.00         1
   Web Developer  

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

jobs = pd.read_csv("../dataset/jobs.csv")

jobs["job_text"] = (
    jobs["required_skills"] + " " + jobs["description"]
)

print("Total jobs:", len(jobs))

# Create the TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    stop_words="english"
)

job_vectors = vectorizer.fit_transform(jobs["job_text"])

def recommend_jobs(resume_text, top_n=5):

    resume_vector = vectorizer.transform([resume_text])

    similarity_scores = cosine_similarity(
        resume_vector,
        job_vectors
    ).flatten()

    jobs_copy = jobs.copy()

    jobs_copy["match_score"] = similarity_scores * 100

    jobs_copy = jobs_copy.sort_values(
        by="match_score",
        ascending=False
    )

    return jobs_copy.head(top_n)[
        ["job_title", "required_skills", "match_score"]
    ]

#Test our recommendation system
resume = """
I am a BCA student with strong skills in Python, SQL,
Pandas, NumPy and Excel. I have worked on data analysis
projects and created dashboards. I also have knowledge
of statistics and Power BI."""

recommendations = recommend_jobs(resume)

print(recommendations)

for index, row in recommendations.iterrows():

    print(
        f"{row['job_title']} "
        f"→ {row['match_score']:.2f}%"
    )

import sys
sys.path.append("..")

from utils.skill_extractor import extract_skills
resume = """
I am a BCA student with strong skills in Python, SQL,
Pandas, NumPy and Excel. I have worked on data analysis
projects and created dashboards. I also have knowledge
of statistics and Power BI.
"""

skills = extract_skills(resume)

print("Detected Skills:")
for skill in skills:
    print("✓", skill)

from utils.pdf_reader import extract_text_from_pdf

pdf_path = "../uploads/Manisha_Danu_Resume.pdf"

resume_text = extract_text_from_pdf(pdf_path)

print(resume_text)

from utils.skill_extractor import extract_skills

skills = extract_skills(resume_text)

print("\nDetected Skills:")
print("-" * 30)

for skill in skills:
    print("✓", skill)

recommendations = recommend_jobs(resume_text)

print("\nTop 5 Job Recommendations")
print("-" * 40)

for index, row in recommendations.iterrows():
    print(
        f"{row['job_title']} → "
        f"{row['match_score']:.2f}%"
    )

Total jobs: 25
           job_title                                    required_skills  \
0       Data Analyst  Python, SQL, Pandas, NumPy, Excel, Power BI, S...   
21        BI Analyst  SQL, Power BI, Tableau, Excel, Data Visualizat...   
20  Business Analyst  SQL, Excel, Power BI, Business Analysis, Commu...   
1     Data Scientist  Python, SQL, Pandas, NumPy, Machine Learning, ...   
22      NLP Engineer  Python, NLP, NLTK, spaCy, Machine Learning, De...   

    match_score  
0     70.171071  
21    51.214339  
20    46.179129  
1     26.896088  
22    11.606232  
Data Analyst → 70.17%
BI Analyst → 51.21%
Business Analyst → 46.18%
Data Scientist → 26.90%
NLP Engineer → 11.61%
Detected Skills:
✓ python
✓ c
✓ sql
✓ pandas
✓ numpy
✓ excel
✓ power bi
✓ statistics
Manisha Danu
Mail: mdanu6763@gmail.com | Contact: +91 70174 25198
LinkedIn: Manisha Danu | GitHub: mdanu7
SUMMARY
Motivated BCA (AI & Data Science) student with knowledge of Python, C programming, basic Data Structures &
Algori

In [6]:
def calculate_skill_gap(resume_skills, required_skills):

    required_skill_list = [
        skill.strip().lower()
        for skill in required_skills.split(",")
    ]

    resume_skill_set = set(
        skill.lower() for skill in resume_skills
    )

    matching_skills = []
    missing_skills = []

    for skill in required_skill_list:

        if skill in resume_skill_set:
            matching_skills.append(skill)
        else:
            missing_skills.append(skill)

    return matching_skills, missing_skills
best_job = recommendations.iloc[0]

print("Recommended Job:", best_job["job_title"])
print("Required Skills:", best_job["required_skills"])

matching, missing = calculate_skill_gap(
    skills,
    best_job["required_skills"]
)

print("\nMatching Skills:")
for skill in matching:
    print("✓", skill)

print("\nMissing Skills:")
for skill in missing:
    print("✗", skill)

skill_match_percentage = (
    len(matching) /
    len(matching + missing)
) * 100

print(
    f"\nSkill Match: "
    f"{skill_match_percentage:.2f}%"
)

Recommended Job: Web Developer
Required Skills: HTML, CSS, JavaScript, Bootstrap, Git, Web Development

Matching Skills:
✓ html
✓ css
✓ javascript
✓ git

Missing Skills:
✗ bootstrap
✗ web development

Skill Match: 66.67%


In [9]:
def calculate_resume_score(
    resume_text,
    detected_skills,
    recommendations
):
    score = 0

    # 1. Skills — 40 points
    skill_score = min(len(detected_skills) * 5, 40)
    score += skill_score

    # 2. Resume length/content — 20 points
    word_count = len(resume_text.split())

    if word_count >= 300:
        content_score = 20
    elif word_count >= 200:
        content_score = 15
    elif word_count >= 100:
        content_score = 10
    else:
        content_score = 5

    score += content_score

    # 3. Job compatibility — 40 points
    if len(recommendations) > 0:
        best_match = recommendations.iloc[0]["match_score"]
        compatibility_score = min(best_match * 0.4, 40)
    else:
        compatibility_score = 0

    score += compatibility_score

    return round(score, 2)
resume_score = calculate_resume_score(
    resume_text,
    skills,
    recommendations
)

print("Resume Score:", resume_score, "/100")

print("\n" + "=" * 50)
print("          AI RESUME ANALYSIS")
print("=" * 50)

print("\nResume Score:")
print(f"{resume_score}/100")

print("\nDetected Skills:")
for skill in skills:
    print("✓", skill)

print("\nTop 5 Job Recommendations:")

for index, row in recommendations.iterrows():
    print(
        f"{row['job_title']} → "
        f"{row['match_score']:.2f}%"
    )

best_job = recommendations.iloc[0]

matching, missing = calculate_skill_gap(
    skills,
    best_job["required_skills"]
)

print("\nBest Matching Job:")
print(best_job["job_title"])

print("\nMatching Skills:")
for skill in matching:
    print("✓", skill)

print("\nMissing Skills:")
for skill in missing:
    print("✗", skill)

print("=" * 50)

Resume Score: 78.39 /100

          AI RESUME ANALYSIS

Resume Score:
78.39/100

Detected Skills:
✓ python
✓ java
✓ c
✓ javascript
✓ data science
✓ excel
✓ html
✓ css
✓ react
✓ rest api
✓ aws
✓ git
✓ figma
✓ ui design

Top 5 Job Recommendations:
Web Developer → 45.98%
Frontend Developer → 36.79%
Full Stack Developer → 32.23%
Python Developer → 20.48%
Business Analyst → 17.27%

Best Matching Job:
Web Developer

Matching Skills:
✓ html
✓ css
✓ javascript
✓ git

Missing Skills:
✗ bootstrap
✗ web development
